# 13. Enkel linjär regression

Korrelationen i förra notebooken svarade på *hur starkt* två variabler hänger ihop. **Regression** går ett steg längre och
beskriver *hur* sambandet ser ut, med en ekvation som kan användas för att göra prognoser:
"hur många poäng ger en extra studietimme?" och "vad kan en student som pluggar 12 timmar vänta sig?"

**Efter den här notebooken kan du**
- förklara vad en regressionslinje är och hur den bestäms (minsta kvadrat-metoden)
- tolka intercept, lutning och förklaringsgrad (R²)
- göra prognoser, och veta när man inte bör göra det
- läsa en residualplot
- förklara varför även regressionskoefficienter har osäkerhet

**Förkunskaper:** [12. Korrelation](../12_korrelation/12_korrelation.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)

# Samma data som i notebook 12: 50 studenters studietid och provresultat
study_hours = rng.uniform(0, 20, 50).round(1)
exam_score = np.clip(40 + 2.5 * study_hours + rng.normal(0, 8, 50), 0, 100).round()

## 13.1 En linje genom punkterna

En rät linje beskrivs av två tal:

$$\hat{y} = a + b \cdot x$$

- **a (intercept):** linjens värde när x = 0.
- **b (lutning, regressionskoefficient):** hur mycket ŷ ändras när x ökar med 1.
- **ŷ** ("y-hatt") är linjens *prognos* för y, till skillnad från det verkliga värdet y.

Innan vi låter datorn räkna: **pröva att lägga linjen själv.**

In [ ]:
a_guess = 30        # ← ändra: var skär linjen y-axeln?
b_guess = 4.0       # ← ändra: hur många poäng per studietimme?

x_line = np.linspace(0, 20, 100)
predicted = a_guess + b_guess * study_hours
residuals = exam_score - predicted
sse = (residuals ** 2).sum()

plt.scatter(study_hours, exam_score)
plt.plot(x_line, a_guess + b_guess * x_line, color="red")
plt.vlines(study_hours, predicted, exam_score, color="gray", alpha=0.5, linewidth=1)
plt.xlabel("Studietimmar")
plt.ylabel("Provpoäng")
plt.title(f"ŷ = {a_guess} + {b_guess} · x     summa av kvadrerade residualer = {sse:.0f}")
plt.show()

De grå strecken är **residualerna**: avståndet i höjdled mellan varje punkt och linjen.

$$\text{residual} = y - \hat{y} = \text{verkligt värde} - \text{prognos}$$

En bra linje har små residualer. För att få *ett* tal på hur bra linjen passar kvadrerar vi residualerna och summerar dem
(kvadraten gör, precis som i standardavvikelsen, att plus och minus inte tar ut varandra). Summan står i rubriken.

🧪 **Labba:** Ändra `a_guess` och `b_guess` och försök få summan så liten som möjligt. Anteckna ditt bästa resultat.
Kommer du under 2 000?

## 13.2 Minsta kvadrat-metoden

**Regressionslinjen** är den linje som ger den *minsta möjliga* summan av kvadrerade residualer. Därav namnet
**minsta kvadrat-metoden**. Det finns en formel för den, så ingen behöver gissa:

In [ ]:
reg = stats.linregress(study_hours, exam_score)

predicted = reg.intercept + reg.slope * study_hours
sse_best = ((exam_score - predicted) ** 2).sum()

print(f"Regressionslinjen: provpoäng = {reg.intercept:.1f} + {reg.slope:.2f} · studietimmar")
print(f"Summa av kvadrerade residualer: {sse_best:.0f}")

Slog du den? Det går inte: ingen annan linje har en lägre summa.

In [ ]:
plt.scatter(study_hours, exam_score, label="Data")
plt.plot(x_line, reg.intercept + reg.slope * x_line, color="red", label="Regressionslinje")
plt.xlabel("Studietimmar")
plt.ylabel("Provpoäng")
plt.legend()
plt.show()

<details>
<summary>📐 <b>För den nyfikne: formlerna bakom</b></summary>

$$b = r \cdot \frac{s_y}{s_x} \qquad\qquad a = \bar{y} - b \cdot \bar{x}$$

Lutningen är alltså korrelationen gånger kvoten mellan standardavvikelserna, och linjen går alltid genom punkten $(\bar{x}, \bar{y})$.
Kontrollera gärna: `stats.pearsonr(study_hours, exam_score)[0] * exam_score.std(ddof=1) / study_hours.std(ddof=1)`.

</details>

## 13.3 Tolka koefficienterna

In [ ]:
print(f"a (intercept) = {reg.intercept:.1f}")
print(f"b (lutning)   = {reg.slope:.2f}")

- **Lutningen b:** varje extra studietimme hänger ihop med i genomsnitt ungefär 2,5 poängs högre resultat.
- **Interceptet a:** en student som inte pluggar alls väntas få ungefär 38 poäng.

Två saker att vara noga med:

1. Säg "hänger ihop med" eller "är förknippad med", inte "ger". Regression mäter samband, och allt i förra notebooken om
   korrelation och kausalitet gäller även här.
2. Interceptet är bara meningsfullt om x = 0 är ett rimligt värde som finns i datan. I en regression av vikt mot längd
   skulle interceptet vara "vikten hos en person som är 0 cm lång".

### Prognoser

In [ ]:
hours = 10          # ← ändra
prediction = reg.intercept + reg.slope * hours
print(f"Prognos för {hours} studietimmar: {prediction:.0f} poäng")

✏️ **Din tur:**
1. Vad är prognosen för 0, 5 och 20 studietimmar?
2. Hur mycket skiljer prognosen mellan en student som pluggar 8 timmar och en som pluggar 12? Kan du svara utan att räkna ut de två prognoserna?
3. Vad är prognosen för 40 timmar? Är den rimlig?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
for h in [0, 5, 20]:
    print(f"{h:>2} timmar: {reg.intercept + reg.slope * h:.1f} poäng")
print(f"Skillnad mellan 12 och 8 timmar: 4 · b = {4 * reg.slope:.1f} poäng")
print(f"40 timmar: {reg.intercept + reg.slope * 40:.1f} poäng")
# Prognosen för 40 timmar är över 100 poäng, vilket är omöjligt på ett prov med maxpoäng 100.
```

</details>

### Varning för extrapolering

Modellen bygger på studenter som pluggade mellan 0 och 20 timmar. Vi vet ingenting om hur sambandet ser ut utanför det
intervallet. Att använda linjen där kallas **extrapolering**, och det kan ge orimliga svar, som i fråga 3.

> Använd regressionslinjen för prognoser **inom** det intervall av x-värden som datan täcker.

## 13.4 Hur bra är modellen? Residualer och R²

En prognos på 64 poäng är inte mycket värd om de verkliga resultaten kan hamna var som helst mellan 20 och 100.
Residualerna visar hur långt ifrån prognoserna brukar hamna.

In [ ]:
residuals = exam_score - (reg.intercept + reg.slope * study_hours)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].scatter(study_hours, residuals)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set_xlabel("Studietimmar")
axes[0].set_ylabel("Residual (poäng)")
axes[0].set_title("Residualplot")
axes[1].hist(residuals, bins=12, edgecolor="white")
axes[1].set_xlabel("Residual (poäng)")
axes[1].set_ylabel("Antal")
axes[1].set_title("Residualernas fördelning")
plt.tight_layout()
plt.show()

print(f"Residualernas standardavvikelse: {residuals.std(ddof=2):.1f} poäng")

Prognoserna missar typiskt med ungefär 6 poäng åt ena eller andra hållet.

I en **residualplot** vill man se en formlös svärm kring noll, som här. Ett mönster betyder att en rät linje inte räcker.

### Förklaringsgraden R²

**R²** anger hur stor andel av variationen i y som modellen förklarar. Jämför två sätt att gissa en students poäng:

- **Utan modell:** gissa medelpoängen för alla. Felet mäts av den totala variationen kring medelvärdet.
- **Med modell:** gissa enligt linjen. Felet är residualerna.

$$R^2 = 1 - \frac{\text{summa av kvadrerade residualer}}{\text{total kvadratsumma kring medelvärdet}}$$

In [ ]:
ss_residual = (residuals ** 2).sum()
ss_total = ((exam_score - exam_score.mean()) ** 2).sum()

print(f"Kvadratsumma utan modell: {ss_total:.0f}")
print(f"Kvadratsumma med modell:  {ss_residual:.0f}")
print(f"R² = {1 - ss_residual / ss_total:.3f}")
print(f"r² = {reg.rvalue ** 2:.3f}   (korrelationskoefficienten i kvadrat)")

Studietiden förklarar alltså ungefär 85 % av variationen i provpoäng. Resten beror på annat: förkunskaper, dagsform, tur.

I enkel linjär regression är R² exakt lika med korrelationskoefficienten i kvadrat. Därav namnet.

### När en rät linje inte passar

In [ ]:
x_curve = rng.uniform(0, 10, 60)
y_curve = 2 + 0.5 * x_curve ** 2 + rng.normal(0, 2, 60)        # ett krökt samband
reg_curve = stats.linregress(x_curve, y_curve)
res_curve = y_curve - (reg_curve.intercept + reg_curve.slope * x_curve)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].scatter(x_curve, y_curve)
xs = np.linspace(0, 10, 50)
axes[0].plot(xs, reg_curve.intercept + reg_curve.slope * xs, color="red")
axes[0].set_title(f"Data och regressionslinje (R² = {reg_curve.rvalue ** 2:.2f})")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")
axes[1].scatter(x_curve, res_curve)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title("Residualplot")
axes[1].set_xlabel("x")
axes[1].set_ylabel("Residual")
plt.tight_layout()
plt.show()

R² är högt, och i den vänstra bilden ser linjen nästan rimlig ut. Men residualplotten avslöjar ett tydligt U-mönster:
linjen ligger för lågt i båda ändar och för högt i mitten. Ett högt R² är alltså ingen garanti för att modellen är rätt.
**Titta alltid på residualerna.**

## 13.5 Osäkerheten i koefficienterna

Vi har facit: datan skapades med a = 40 och b = 2,5, plus slumpmässigt brus. Regressionen gav nästan, men inte exakt, de värdena.
Skälet är detsamma som i [notebook 5](../05_population_och_stickprov/05_population_och_stickprov.ipynb):
vi har bara ett stickprov på 50 studenter.

🔮 **Gissa först:** om vi drar 1 000 nya stickprov om 50 studenter, mellan vilka värden hamnar lutningen?

In [ ]:
n_students = 50        # ← ändra: pröva 10 och 500

slopes = []
for _ in range(1000):
    hours_sim = rng.uniform(0, 20, n_students)
    score_sim = 40 + 2.5 * hours_sim + rng.normal(0, 8, n_students)
    slopes.append(stats.linregress(hours_sim, score_sim).slope)

plt.hist(slopes, bins=30, edgecolor="white")
plt.axvline(2.5, color="red", linestyle="--", label="sann lutning 2,5")
plt.xlabel("Skattad lutning b")
plt.ylabel("Antal stickprov")
plt.title(f"1 000 stickprov med n = {n_students}  (spridning {np.std(slopes):.3f})")
plt.legend()
plt.show()

Lutningen har en samplingfördelning, precis som ett medelvärde. Dess standardavvikelse är lutningens **medelfel**, och
`linregress` skattar det från vårt enda stickprov:

In [ ]:
print(f"b = {reg.slope:.3f}")
print(f"Medelfel för b: {reg.stderr:.3f}   (jämför med spridningen i simuleringen för n = 50)")
print(f"p-värde för H0: b = 0:  {reg.pvalue:.2g}")

p-värdet testar $H_0$: lutningen är 0 i populationen, alltså *inget linjärt samband*. Det är samma test som
korrelationens p-värde i förra notebooken.

✏️ **Din tur:** Räkna ut ett 95 % konfidensintervall för lutningen: $b \pm t^* \cdot \text{medelfel}$, där t\* har n − 2
frihetsgrader (`stats.t.ppf(0.975, df=len(study_hours) - 2)`). Täcker intervallet det sanna värdet 2,5? Innehåller det 0?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
t_crit = stats.t.ppf(0.975, df=len(study_hours) - 2)
low, high = reg.slope - t_crit * reg.stderr, reg.slope + t_crit * reg.stderr
print(f"95 % KI för b: [{low:.2f}, {high:.2f}]")
# Intervallet täcker 2,5 och ligger långt från 0, vilket stämmer med det mycket låga p-värdet.
```

</details>

## 13.6 Öva på ny data

✏️ **Din tur:** En bilhandlare har noterat ålder (år) och pris (tkr) för 12 begagnade bilar av samma modell:

```python
age = np.array([1, 2, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10])
price = np.array([245, 228, 235, 205, 190, 172, 180, 160, 148, 130, 122, 105])
```

1. Rita ett spridningsdiagram.
2. Anpassa en regressionslinje. Tolka lutningen och interceptet i ord.
3. Vad är R²?
4. Vad är prognosen för en bil som är 6 år? 25 år? Kommentera.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
age = np.array([1, 2, 2, 3, 4, 5, 5, 6, 7, 8, 9, 10])
price = np.array([245, 228, 235, 205, 190, 172, 180, 160, 148, 130, 122, 105])

reg_car = stats.linregress(age, price)
plt.scatter(age, price)
plt.plot(age, reg_car.intercept + reg_car.slope * age, color="red")
plt.xlabel("Ålder (år)")
plt.ylabel("Pris (tkr)")
plt.show()

print(f"pris = {reg_car.intercept:.1f} + ({reg_car.slope:.1f}) · ålder,   R² = {reg_car.rvalue ** 2:.3f}")
print(f"6 år:  {reg_car.intercept + reg_car.slope * 6:.0f} tkr")
print(f"25 år: {reg_car.intercept + reg_car.slope * 25:.0f} tkr")
# Lutning: priset sjunker med i genomsnitt knappt 16 tkr per år.
# Intercept: en ny bil (0 år) väntas kosta ungefär 257 tkr – en lätt extrapolering, eftersom den yngsta bilen är 1 år.
# Prognosen för 25 år är negativ: extrapolering långt utanför datan ger nonsens.
```

</details>

## Sammanfattning

- Regressionslinjen **ŷ = a + b · x** är den linje som minimerar summan av kvadrerade residualer.
- **b** = genomsnittlig förändring i y när x ökar med 1. **a** = förväntat y när x = 0.
- **R²** = andelen av variationen i y som modellen förklarar (= r² i enkel linjär regression).
- **Residualplotten** visar om en rät linje är en rimlig modell.
- Koefficienterna är skattningar med **medelfel**, konfidensintervall och p-värden.
- **Extrapolera inte**, och kom ihåg att regression visar samband, inte orsak.

### Kontrollfrågor
1. En regression av lön (tkr) mot antal år i yrket ger ŷ = 28 + 0,9 · x. Tolka 28 och 0,9.
2. Två regressioner har samma lutning, men R² är 0,9 i den ena och 0,2 i den andra. Vad är skillnaden i praktiken?
3. Lutningen i en regression har p = 0,40. Vad betyder det?

<details>
<summary>💡 <b>Svar</b></summary>

1. En nyanställd (0 år) väntas tjäna 28 tkr. Varje extra år i yrket är förknippat med i genomsnitt 0,9 tkr högre lön.
2. Sambandet är i genomsnitt lika brant, men i den andra ligger punkterna mycket mer utspridda kring linjen. Prognoser för
   enskilda individer blir mycket osäkrare.
3. Datan ger inga belägg för att lutningen skiljer sig från 0. Vi kan inte påvisa något linjärt samband
   (vilket inte bevisar att det saknas).

</details>

**Nästa notebook:** [14. Från Python till R](../14_python_och_r/14_python_och_r.ipynb)